In [1]:
OPENAI_KEY = "INSERT_KEY"
GEMINI_KEY = "INSERT_KEY"
CLAUDE_KEY = "INSERT_KEY"

In [5]:
import os, re
import pandas as pd
from pathlib import Path
from docx import Document

# --- 1. FUNCTIONS ---

def _read_docx(path: Path) -> str:
    try:
        doc = Document(path)
        return "\n".join([para.text for para in doc.paragraphs]).strip()
    except:
        return ""

def _read_dir_texts(folder: Path) -> dict:
    """
    Extracts the essay ID by taking the number at the VERY END of the filename.
    Example: 'draft1_2018argrewrite_3.txt' -> ID is '3'
    """
    texts = {}
    if not folder.is_dir(): return texts
    for fn in os.listdir(folder):
        if not fn.lower().endswith(".txt"): continue
        
        # This regex looks for digits immediately followed by the file extension
        # Example: '..._3.txt' matches '3'
        m = re.search(r"(\d+)\.txt$", fn)
        if m:
            eid = m.group(1)
            with open(folder / fn, "r", encoding="utf-8", errors="ignore") as f:
                texts[eid] = f.read().strip()
    return texts

def _read_feedback_dir(folder: Path) -> dict:
    """
    Extracts the ID from feedback files.
    Example: 'feedback_1.docx' -> ID is '1'
    """
    fb_texts = {}
    if not folder.is_dir(): return fb_texts
    for fn in os.listdir(folder):
        if not fn.endswith(".docx") or fn.startswith("~$"): continue
        
        # This looks for any number in 'feedback_1.docx'
        m = re.search(r"(\d+)", fn)
        if m:
            eid = m.group(1)
            content = _read_docx(folder / fn)
            if content:
                fb_texts[eid] = content
    return fb_texts

# --- 2. PATHS ---
ARGREWRITE_ROOT = Path("../ArgRewrite").resolve()
ESSAYS_DIR = ARGREWRITE_ROOT / "dataset" / "essays"
FEEDBACK_DIR = ARGREWRITE_ROOT / "dataset" / "meta-data" / "ExpertFeedback"

D1, D2 = ESSAYS_DIR / "Draft1", ESSAYS_DIR / "Draft2"

# --- 3. EXECUTE ---
print("Loading Drafts...")
draft1 = _read_dir_texts(D1)
draft2 = _read_dir_texts(D2)

print("Loading Feedback...")
feedback_map = _read_feedback_dir(FEEDBACK_DIR)

# Intersection of IDs found in D1, D3, and Feedback
# We only want to process essays that have all three components
paired_ids = sorted(set(draft1) & set(draft2) & set(feedback_map), key=lambda x: int(x))

df = pd.DataFrame([{
    "essay_id": f"argrewrite_{eid}",
    "raw_id": eid,
    "draft1": draft1[eid],
    "draft2": draft2[eid],
    "feedback_text": feedback_map[eid]
} for eid in paired_ids])

print("-" * 30)
print(f"✅ Success! Found {len(df)} essays where Draft1, Draft2, and Feedback all match.")

Loading Drafts...
Loading Feedback...
------------------------------
✅ Success! Found 86 essays where Draft1, Draft2, and Feedback all match.


In [ ]:
import os, re, time
import pandas as pd
from pathlib import Path
from docx import Document
import numpy as np

# --- API CLIENTS ---
import openai
from anthropic import Anthropic
import google.generativeai as genai

client_oa = openai.OpenAI(api_key=OPENAI_KEY)
client_ant = Anthropic(api_key=CLAUDE_KEY)
genai.configure(api_key=GEMINI_KEY)

# --- CONFIGURATION ---
STRICT_SYSTEM_PROMPT = (
    "You are a helpful AI assistant."
)

PROMPT_VARIANTS = {
    "expert": "You are given an essay draft. Revise the essay.",
    "minimal": "You are given an essay draft. Rewrite this essay and keep the original meaning.",
    "minimal": "You are given an essay draft. Rewrite this essay and keep to a similar word count.",
    "grammar": "You are given an essay draft. Revise this essay for grammar.",
    "completion": "You are given an essay draft. The text is only the first paragraph. Finish the rest of the essay.",
    "expansion": "You are given an essay draft. Expand on the ideas in the following draft with more detail and depth.",
}

# Add models back here to run them all
MODELS = [
    {"provider": "openai-reasoning", "name": "gpt-5-mini"},
    # {"provider": "openai", "name": "gpt-4o"},
    # {"provider": "anthropic", "name": "claude-haiku-4-5-20251001"}
    # {"provider": "gemini", "name": "gemini-2.5-flash"} # The stable production choice]
]
# ---------------------------
# 1. DATA LOADERS
# ---------------------------
def _read_dir_texts(folder: Path, prefix_to_strip: str = "") -> dict:
    texts = {}
    if not folder.is_dir(): return texts
    for fn in os.listdir(folder):
        if not fn.lower().endswith(".txt"): continue
        m = re.search(r"(\d+)", fn[len(prefix_to_strip):] if fn.startswith(prefix_to_strip) else fn)
        if m:
            with open(folder / fn, "r", encoding="utf-8", errors="ignore") as f:
                texts[m.group(1)] = f.read()
    return texts

def _read_docx_and_clean(path: Path) -> str:
    try:
        doc = Document(path)
        full_text = "\n".join([para.text for para in doc.paragraphs])
        trigger = "We advise that you use this feedback when you revise"
        if trigger in full_text:
            cleaned_text = full_text.split(trigger)[-1].strip()
            return re.sub(r'^[:\s\.]+', '', cleaned_text)
        return full_text.strip()
    except: return ""

def _read_feedback_dir(folder: Path) -> dict:
    fb_texts = {}
    if not folder.is_dir(): return fb_texts
    for fn in os.listdir(folder):
        if not fn.endswith(".docx") or fn.startswith("~$"): continue
        m = re.search(r"(\d+)", fn)
        if m: fb_texts[m.group(1)] = _read_docx_and_clean(folder / fn)
    return fb_texts

# ---------------------------
# 2. LLM CALL ROUTER
# ---------------------------
def call_llm(text, feedback, provider, model_name, prompt_instruction):
    user_content = f"{prompt_instruction}\n\n"
    if feedback:
        user_content += f"Incorporate the Expert Feedback below:\n{feedback}\n\n"
    user_content += f"CURRENT DRAFT:\n{text}"
    user_content += f"\n\nOnly provide the edited essay and no other feedback.\n\nEdited Essay:\n"

    print("Instructions")
    print(user_content)
    if provider == "openai":
        resp = client_oa.chat.completions.create(
            model=model_name,
            messages=[{"role": "system", "content": STRICT_SYSTEM_PROMPT}, {"role": "user", "content": user_content}],
            temperature=0.7
        )
        return resp.choices[0].message.content.strip()
    if provider == "openai-reasoning":
        ret = {"thoughts": "", "response": ""}
        prompt = [
            {"role": "system", "content": STRICT_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
        
    
        response = client_oa.responses.create(
            model="gpt-5-mini",
            reasoning={
                "effort": "low",   # low | medium | high
                "summary": "auto"     # auto | concise | detailed
            },
            input=prompt
        )
    
        summary = ""
        answer = ""
    
        for item in response.output:
            if item.type == "reasoning" and item.summary:
                summary = item.summary[0].text
            elif item.type == "message" and item.content:
                answer = item.content[0].text
    
        ret["thoughts"] = summary
        ret["response"] = answer
        return ret["response"]

    elif provider == "anthropic":
        resp = client_ant.messages.create(
            model=model_name,
            system=STRICT_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_content}],
            temperature=0.7,
            max_tokens=8192)  # Changed from 4096
        return resp.content[0].text.strip()

    elif provider == "gemini":
            try:
                clean_name = model_name.replace("models/", "")
                model = genai.GenerativeModel(
                    model_name=clean_name,
                    system_instruction=STRICT_SYSTEM_PROMPT
                )
                
                # Use generation_config to force a higher limit
                response = model.generate_content(
                    user_content,
                    generation_config=genai.types.GenerationConfig(
                        max_output_tokens=8192,  # Set to max for full essays
                        temperature=0.7,
                        top_p=0.95,
                    )
                )
                return response.text.strip()
                
            except Exception as e:
                print(f"⚠️ Gemini Error: {e}")
                raise e

# ---------------------------
# 4. EXECUTION LOOP
# ---------------------------
for m_cfg in MODELS:
    provider = m_cfg["provider"]
    model_name = m_cfg["name"]
    # Sanitize name for file system (e.g., gpt-4o-mini)
    safe_model_name = model_name.split('/')[-1]

    # NEW: Create a dedicated directory for this model's outputs
    model_folder = Path(f"llm_drafts_{safe_model_name}")
    model_folder.mkdir(parents=True, exist_ok=True)

    for p_slug, p_instr in PROMPT_VARIANTS.items():
        for use_fb in [True, False]:
            suffix = "with_FB" if use_fb else "no_FB"
            
            # Updated filename to include the new folder path
            fname = model_folder / f"llm_draft_{safe_model_name}_{p_slug}_{suffix}.csv"

            # Check if we already finished this specific test
            if fname.exists():
                print(f"⏩ Skipping {fname.name} (Already exists)")
                continue

            print(f"🚀 Running {safe_model_name} | {p_slug} | {suffix}")
            res_df = df.copy()
            res_df["output_text"] = None
            processed_count = 0

            for i, row in res_df.iterrows():
                fb = row["feedback_text"] if use_fb else None
                
                # Skip if we are doing a "With Feedback" run but no feedback exists
                if use_fb and (not fb or str(fb).strip() == ""):
                    continue 

                try:
                    # Your LLM call router
                    output = call_llm(row["draft1"], fb, provider, model_name, p_instr)
                    res_df.at[i, "output_text"] = output
                    print("OUTPUT")
                    print(output)
                    processed_count += 1
                    
                    # Log progress for long runs
                    if processed_count % 5 == 0:
                        print(f"   ... {processed_count} essays completed")
                    
                    time.sleep(0.5) 
                except Exception as e:
                    print(f"   ❌ Error on {row['essay_id']}: {e}")

                # Save only if we actually generated content
                if processed_count > 0:
                    final_output = res_df.dropna(subset=["output_text"])
                    final_output.to_csv(fname, index=False)
                    print(f"✅ Saved {processed_count} rows to {fname}")
                else:
                    print(f"ℹ️ No rows processed for {fname.name}")
                    

/tmp/ipykernel_123613/4281474113.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


🚀 Running gpt-5-mini | expert | with_FB
Instructions
You are given an essay draft. Revise the essay.

Incorporate the Expert Feedback below:
Thank you for your participation in the study. Your draft has been read, and feedback from an expert writing instructor is written below. We advise that you use this feedback when you revise. 

The strengths of your essay include:
You provided specific and convincing evidence for each claim, and most evidence is given through relevant direct quotations or detailed examples from the provided reading. 
You make multiple, distinct claims that are clear, and align with both your thesis statement, and the given reading. All of your claims are easily located in each paragraph and fully support your argument.
You essay has a logical sequence of ideas, and each body paragraph makes a distinct claim. Your essay has an introduction, body, and conclusion. 
You responded to all parts of the prompt, and the entire essay is focused on the prompt.		
Areas to imp